## Section 1. Introduction to the Problem/Task

### Problem Statement
Medical professionals and researchers frequently need to query large, complex clinical guideline documents such as the **2024 European Society of Cardiology (ESC) Guidelines**. These documents span hundreds of pages and contain dense, technical information covering diagnostic criteria, treatment algorithms, risk stratification scores, and drug recommendations.

Manually searching through such documents is time-consuming and error-prone. This project addresses that challenge by building a **Retrieval-Augmented Generation (RAG) system** that allows users to ask natural language questions and receive accurate, context-grounded answers — complete with source page citations.

### Approach
This notebook implements a full RAG pipeline using **Microsoft Phi-3-Mini-4k-Instruct** (a compact but capable 3.8B parameter model optimized for instruction following) as the language model backbone:

1. **PDF Ingestion** — The 2024 ESC Guidelines PDF is parsed and chunked into overlapping text segments.
2. **Embedding** — Chunks are embedded using **MedCPT** (a biomedical bi-encoder) and stored in a **FAISS** vector index.
3. **Retrieval & Reranking** — At query time, the top candidate chunks are retrieved and reranked using a **Cross-Encoder** for precision.
4. **Generation** — The reranked context is fed to the LLM to produce a final answer.
5. **Evaluation** — Responses are scored using ROUGE, BERTScore, Semantic Similarity, and LLM-as-a-judge metrics.
6. **Deployment** — The system is served through an interactive **Gradio** web interface.

## Section 2. Dataset Description

### Source Document
The dataset used in this project is the **2024 ESC Guidelines on Atrial Fibrillation** (and related cardiovascular conditions), published by the European Society of Cardiology. It is a comprehensive clinical reference document containing:

- **Treatment recommendations** classified by evidence level (Class I, IIa, IIb, III)
- **Diagnostic criteria** for cardiovascular conditions
- **Risk scores** (CHA₂DS₂-VASc, HAS-BLED, etc.)
- **Drug dosing tables** and contraindication summaries
- **Structured algorithms** for clinical decision-making

The PDF is dense, multi-columnar, and includes both prose and tabular data — making it a challenging but representative real-world retrieval task.

### 2.1 Dataset Cleaning

The raw PDF is processed page-by-page using a table-aware parser. Text and tabular content are extracted separately and then combined per page. An exported `.txt` file is also saved for inspection and reproducibility.

In [1]:
# Install core libraries
!pip install -q -U "transformers>=4.41.2" accelerate bitsandbytes langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pdfplumber

# Install UI and metrics libraries
!pip install -q gradio
!pip install -q rouge-score bert-score nltk pandas

print("-" * 50)
print("✅ ALL LIBRARIES INSTALLED")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take in

In [2]:
import os, csv, time, torch, gc, pdfplumber, warnings, re, string, collections
import pandas as pd
from datetime import datetime
from typing import List
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

warnings.filterwarnings("ignore")

# ==========================================
# PDF LOADER (TABLE-AWARE WITH TEXT EXPORT)
# ==========================================
def load_pdf_with_tables(path, output_filename="extracted_pdf_data.txt"):
    print(f"📂 Loading {path}...")
    documents = []

    if not os.path.exists(path):
        raise FileNotFoundError(f"File {path} not found. Please upload it to Colab files.")

    with pdfplumber.open(path) as pdf:
        total_pages = len(pdf.pages)
        for i, page in enumerate(pdf.pages):
            if i % 10 == 0: print(f"Processing page {i}/{total_pages}...")
            tables = page.extract_tables()
            table_str = ""
            if tables:
                for table in tables:
                    if not table: continue
                    table_str += "\n\n**[TABLE DATA]**\n"
                    for row in table:
                        clean_row = [str(cell).replace("\n", " ") if cell else "" for cell in row]
                        table_str += "| " + " | ".join(clean_row) + " |\n"
            text = page.extract_text() or ""
            full_content = text + "\n" + table_str
            documents.append(Document(page_content=full_content, metadata={"page": i+1}))

    print(f"💾 Saving extracted data to {output_filename}...")
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("=== RAG PIPELINE EXTRACTED DATA ===\n")
        f.write(f"Source: {path}\n")
        f.write("====================================\n\n")
        for doc in documents:
            f.write(f"--- PAGE {doc.metadata['page']} ---\n")
            f.write(doc.page_content + "\n\n")

    print(f"✅ Loaded {len(documents)} pages.")
    return documents

### 2.2 Comparing Embedding Models

For document and query embedding, we use **MedCPT** (Medical Contrastive Pre-Training), a biomedical bi-encoder from NCBI trained on PubMed data. It uses separate encoders for queries and documents, making it well-suited for asymmetric retrieval tasks like this one.

MedCPT was chosen over general-purpose models (e.g., `all-MiniLM-L6-v2`) because of its domain-specific pretraining on medical literature, which produces embeddings that better capture clinical terminology and concept relationships.

In [3]:
# ==========================================
# MEDCPT EMBEDDINGS (BI-ENCODER)
# ==========================================
class MedCPTEmbeddings(Embeddings):
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"⚙️ Initializing MedCPT on {self.device}...")
        self.article_tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
        self.article_model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder").to(self.device)
        self.query_tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
        self.query_model = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder").to(self.device)

    def embed_documents(self, texts):
        all_embeddings = []
        batch_size = 16
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            with torch.no_grad():
                encoded = self.article_tokenizer(batch, max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
                output = self.article_model(**encoded)
                embeddings = output.last_hidden_state[:, 0, :]
                all_embeddings.extend(embeddings.cpu().tolist())
                del encoded, output, embeddings
                torch.cuda.empty_cache()
        return all_embeddings

    def embed_query(self, text):
        with torch.no_grad():
            encoded = self.query_tokenizer([text], max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
            output = self.query_model(**encoded)
            return output.last_hidden_state[:, 0, :][0].cpu().tolist()

    def free_article_encoder(self):
        print("🧹 Optimizing Memory: Removing Article Encoder...")
        del self.article_model, self.article_tokenizer
        torch.cuda.empty_cache()
        gc.collect()

# ==========================================
# INGESTION & VECTOR STORE
# ==========================================
pdf_path = "/content/2024ESC-compressed.pdf"
raw_docs = load_pdf_with_tables(pdf_path, output_filename="extracted_pdf_data.txt")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(raw_docs)

medcpt = MedCPTEmbeddings()
print("🧠 Embedding Documents...")
vectorstore = FAISS.from_documents(splits, medcpt)
medcpt.free_article_encoder()

📂 Loading /content/2024ESC-compressed.pdf...
Processing page 0/101...
Processing page 10/101...
Processing page 20/101...
Processing page 30/101...
Processing page 40/101...
Processing page 50/101...
Processing page 60/101...
Processing page 70/101...
Processing page 80/101...
Processing page 90/101...
Processing page 100/101...
💾 Saving extracted data to extracted_pdf_data.txt...
✅ Loaded 101 pages.
⚙️ Initializing MedCPT on cuda...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🧠 Embedding Documents...
🧹 Optimizing Memory: Removing Article Encoder...


## Section 3. Requirements

The following libraries are required to run this notebook. They cover deep learning (PyTorch, Transformers), RAG pipeline components (LangChain, FAISS), PDF parsing, and evaluation metrics (ROUGE, BERTScore, Gradio UI).

In [4]:
# Install core libraries
#!pip install -q -U "transformers>=4.41.2" accelerate bitsandbytes langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pdfplumber

# Install UI and metrics libraries
#!pip install -q gradio
#!pip install -q rouge-score bert-score nltk pandas


## Section 4. System Architecture

The RAG pipeline is composed of four core components:

1. **Vector Store (FAISS)** — Stores MedCPT embeddings of document chunks for fast approximate nearest-neighbor retrieval.
2. **Cross-Encoder Reranker** — Re-scores the top-k retrieved chunks using `cross-encoder/ms-marco-MiniLM-L-6-v2`.
3. **Language Model (LLM)** — Phi-3-Mini-4k-Instruct generates the final answer conditioned on reranked context.
4. **Semantic Similarity Model** — `all-MiniLM-L6-v2` is loaded on CPU for offline metric computation.

All models are loaded with **4-bit quantization (NF4)** via BitsAndBytes to fit within VRAM constraints.

In [5]:
print("⚖️ Loading Cross-Encoder (Reranker)...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')

print("⚙️ Loading Semantic Similarity Model...")
metric_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

print("🚀 Loading Phi-3 Mini...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=False,
    device_map="auto"
)
print("✅ Phi-3 Mini ready.")

⚖️ Loading Cross-Encoder (Reranker)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Phi-3 Mini ready.


## Section 5. System Evaluation (Unseen Queries)

The system is evaluated using a multi-dimensional set of metrics to capture different aspects of response quality:

| Metric Group | Metrics | Description |
|---|---|---|
| **Lexical** | ROUGE-1, ROUGE-2, ROUGE-L | N-gram overlap with reference answer |
| **Semantic** | BERTScore (P/R/F1), Cosine Similarity | Embedding-space similarity |
| **RAG-specific** | Faithfulness, Answer Relevancy, Context Recall | LLM-as-a-judge (0–10 scale) |

These metrics collectively assess whether responses are factually grounded, relevant to the query, and linguistically similar to expected answers.

In [6]:
def normalize_text(s):
    if pd.isna(s): return ""
    s = str(s).lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def calc_accuracy_metrics(pred, truth):
    norm_pred = normalize_text(pred).split()
    norm_truth = normalize_text(truth).split()
    exact_match = int(norm_pred == norm_truth)
    common = collections.Counter(norm_pred) & collections.Counter(norm_truth)
    num_same = sum(common.values())
    if len(norm_pred) == 0 or len(norm_truth) == 0:
        f1 = precision = recall = int(norm_pred == norm_truth)
    elif num_same == 0:
        f1 = precision = recall = 0.0
    else:
        precision = num_same / len(norm_pred)
        recall = num_same / len(norm_truth)
        f1 = (2 * precision * recall) / (precision + recall)
    return exact_match, round(precision, 4), round(recall, 4), round(f1, 4)

def calc_lexical_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0, 0, 0, 0, 0, 0
    smoothie = SmoothingFunction().method4
    ref = [normalize_text(truth).split()]
    hyp = normalize_text(pred).split()
    b1 = sentence_bleu(ref, hyp, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    b2 = sentence_bleu(ref, hyp, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    b4 = sentence_bleu(ref, hyp, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(str(truth), str(pred))
    return round(b1,4), round(b2,4), round(b4,4), round(rouge_scores['rouge1'].fmeasure,4), round(rouge_scores['rouge2'].fmeasure,4), round(rouge_scores['rougeL'].fmeasure,4)

def calc_semantic_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0,0,0,0
    P, R, F1 = bert_score([str(pred)], [str(truth)], lang="en", model_type="distilbert-base-uncased", device="cpu")
    embeddings = metric_model.encode([str(pred), str(truth)], convert_to_tensor=True)
    cosine_sim = float(util.pytorch_cos_sim(embeddings[0], embeddings[1])[0][0])
    return round(P.item(),4), round(R.item(),4), round(F1.item(),4), round(cosine_sim,4)

def calc_rag_metrics_llm(query, context, response, expected):
    messages = [
        {"role": "user", "content": (
            "You are a strict grading assistant. Grade the Response against the Context and Expected Answer.\n\n"
            f"Context: {context[:1500]}\nQuery: {query}\nExpected Answer: {expected}\nResponse: {response}\n\n"
            "Output exactly three numbers from 0 to 10 in this exact format: [FTH, REL, CTX]\n"
            "- FTH (Faithfulness): Is the Response supported by the Context?\n"
            "- REL (Relevancy): Does the Response answer the Query?\n"
            "- CTX (Recall): Is the Expected Answer in the Context?\n"
            "Do NOT output any other text or explanations."
        )}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=25, do_sample=False)
    input_length = input_ids.shape[1]
    output = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()
    del input_ids, output_ids
    torch.cuda.empty_cache()
    try:
        numbers = re.findall(r'\d+', output)
        if len(numbers) >= 3:
            return min(int(numbers[0]), 10), min(int(numbers[1]), 10), min(int(numbers[2]), 10)
    except:
        pass
    return 0, 0, 0

In [7]:
def process_query_stream(query, expected=""):
    yield "⏳ **Status:** 🔍 Retrieving relevant documents from VectorDB...\n\n---\n"
    retrieval_start_time = time.time()
    retrieved_docs = vectorstore.similarity_search(query, k=60)

    unique_docs = []
    seen_pages = set()
    for doc in retrieved_docs:
        page = doc.metadata.get('page')
        if page not in seen_pages:
            unique_docs.append(doc)
            seen_pages.add(page)

    yield "⏳ **Status:** 📊 Reranking top documents with CrossEncoder...\n\n---\n"
    pairs = [[query, doc.page_content] for doc in unique_docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(unique_docs, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, score in scored_docs[:5]]

    context_text = "\n\n".join([d.page_content for d in top_docs])
    source_pages = ", ".join([str(d.metadata.get('page')) for d in top_docs])

    yield "⏳ **Status:** 🧠 Synthesizing answer with Phi-3 (this may take a moment)...\n\n---\n"
    messages = [
        {"role": "system", "content": (
            "You are a strict clinical assistant for the ESC Guidelines. Answer based ONLY on the context. "
            "CRITICAL RULES:\n1. Define scoring systems strictly.\n2. State clearly if a treatment is Class III.\n"
            "3. Do not hallucinate acronym meanings.\n"
            "4. If the prompt is not medical, output: 'This model only answers to medical questions.'"
        )},
        {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion:\n{query}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    generation_start_time = time.time()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids, max_new_tokens=300, do_sample=False,
            repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
        )
    generation_time = time.time() - generation_start_time
    input_length = input_ids.shape[1]
    generated_tokens = len(output_ids[0]) - input_length
    tokens_per_sec = generated_tokens / generation_time if generation_time > 0 else 0
    clean_response = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()
    del input_ids, output_ids
    torch.cuda.empty_cache()

    yield "⏳ **Status:** 📈 Calculating evaluation metrics...\n\n---\n"
    exact_match, tok_p, tok_r, tok_f1 = calc_accuracy_metrics(clean_response, expected)
    bleu1, bleu2, bleu4, rouge1, rouge2, rougeL = calc_lexical_metrics(clean_response, expected)
    bert_p, bert_r, bert_f1, cos_sim = calc_semantic_metrics(clean_response, expected)
    faithfulness, answer_rel, ctx_recall = calc_rag_metrics_llm(query, context_text, clean_response, expected)

    metadata = f"**Source Pages:** {source_pages}\n\n"
    metrics_text = (f"**Latency:** {round(generation_time, 2)}s | "
                    f"**Speed:** {round(tokens_per_sec, 2)} Tokens/sec | "
                    f"**RAG Scores (F/R/C):** {faithfulness}/10, {answer_rel}/10, {ctx_recall}/10")
    yield f"{clean_response}\n\n---\n{metadata}{metrics_text}"

## Section 6. User Interface (Gradio)

The system is wrapped in an interactive web UI built with **Gradio**. Users can type clinical questions and receive streamed responses with real-time status updates for each pipeline stage (retrieval → reranking → generation → evaluation).

The interface uses a **Teal/Blue** Phi-3 theme.

In [8]:
import gradio as gr

def gradio_wrapper(query):
    if not query or not query.strip():
        yield "⚠️ Please enter a valid question."
        return
    for status_update in process_query_stream(query, expected=""):
        yield status_update

phi_theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont('Inter'), 'ui-sans-serif', 'system-ui', 'sans-serif']
).set(
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_700",
    block_title_text_color="*primary_700",
    block_label_text_color="*primary_600"
)

with gr.Blocks(theme=phi_theme) as demo:
    gr.Markdown("# ⚕️ Cardiology AI Assistant (ESC 2024)")
    gr.Markdown("### ⚡ Powered by Microsoft Phi-3-Mini")
    gr.Markdown("Ask questions based on the 2024 ESC Medical Guidelines. The system uses RAG with MedCPT embeddings, Cross-Encoder reranking, and Phi-3 generation.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Your Clinical Question",
                placeholder="e.g., What are the class I recommendations for anticoagulation in AF?",
                lines=3
            )
            submit_btn = gr.Button("Analyze Guidelines", variant="primary")

    output_text = gr.Markdown(label="Assistant Response")

    gr.Examples(
        examples=[
            "What are the class I recommendations for anticoagulation in AF?",
            "Summarize the treatment algorithm for chronic heart failure.",
            "What is the target LDL-C for very high-risk patients?"
        ],
        inputs=input_text
    )
    submit_btn.click(gradio_wrapper, inputs=input_text, outputs=output_text)

### Section 6.1 Deployed Web Application

The Gradio app is launched with `share=True` to generate a public Hugging Face Spaces-style URL. Queue mode is enabled to support streaming (yielded) responses.

In [9]:
demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1fef60229896c6870a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1fef60229896c6870a.gradio.live


## Section 6.2 Evaluation Results — Per-Query Data

The table below presents the full per-query evaluation results for **Phi-3-Mini-4k-Instruct** across all 20 test queries.

| Query | Pages | ROUGE-1 | ROUGE-2 | ROUGE-L | BERTScore P | BERTScore R | BERTScore F1 | Semantic Similarity | Faithfulness | Answer Relevancy | Context Recall | Retrieval Time (s) | Generation Time (s) | Tokens/sec |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| What are the four essential treatment pillars of the AF-CARE framework? | 19, 9, 25, 3, 26 | 0.1918 | 0.0563 | 0.0822 | 0.7402 | 0.7445 | 0.7424 | 0.3114 | 9 | 10 | 7 | 0.4035 | 12.92 | 5.96 |
| What is the minimum ECG duration required to establish a diagnosis of clinical AF? | 15, 12, 11, 60, 66 | 0.325 | 0.1795 | 0.175 | 0.7579 | 0.8573 | 0.8046 | 0.673 | 9 | 10 | 10 | 0.1908 | 14.39 | 6.6 |
| When is oral anticoagulation (OAC) recommended based on the CHA2DS2-VA score? | 47, 9, 68, 92, 29 | 0.2158 | 0.0876 | 0.1871 | 0.6704 | 0.8141 | 0.7353 | 0.4162 | 10 | 10 | 10 | 0.2037 | 21.44 | 8.96 |
| Which drugs are recommended as first-choice for rate control in patients with LVEF>40%? | 67, 10, 38, 21, 39 | 0.3636 | 0.3226 | 0.3636 | 0.8029 | 0.9294 | 0.8615 | 0.6967 | 10 | 10 | 10 | 0.2155 | 10.04 | 4.98 |
| Is antiplatelet therapy recommended as an alternative to OAC for stroke prevention in AF? | 66, 67, 33, 20, 9 | 0.1835 | 0.1121 | 0.1651 | 0.757 | 0.8073 | 0.7813 | 0.7723 | 10 | 10 | 10 | 0.1889 | 17.29 | 8.62 |
| What is the recommended target for alcohol consumption to reduce AF recurrence? | 12, 66, 25, 26, 63 | 0.7273 | 0.5806 | 0.6667 | 0.8734 | 0.9183 | 0.8953 | 0.7354 | 10 | 10 | 10 | 0.2248 | 8.65 | 3.93 |
| For patients on VKA, what is the recommended target INR range and TTR percentage? | 67, 31, 53, 52, 32 | 0.48 | 0.25 | 0.48 | 0.829 | 0.8839 | 0.8556 | 0.587 | 10 | 10 | 10 | 0.2253 | 12.69 | 4.88 |
| When should a "wait-and-see" approach be considered for cardioversion? | 10, 40, 41, 42, 56 | 0.3051 | 0.0351 | 0.1695 | 0.7915 | 0.8437 | 0.8168 | 0.5992 | 9 | 10 | 10 | 0.2202 | 11.5 | 6.17 |
| List the three criteria used to decide on a dose reduction for Apixaban. | 32, 64, 38, 31, 52 | 0.52 | 0.2917 | 0.44 | 0.8452 | 0.8337 | 0.8394 | 0.5763 | 10 | 10 | 10 | 0.1616 | 11.85 | 4.81 |
| What is the primary indication for long-term rhythm control therapy in AF patients? | 64, 45, 44, 39, 42 | 0.4138 | 0.3571 | 0.3793 | 0.8171 | 0.8915 | 0.8527 | 0.6277 | 10 | 10 | 10 | 0.2039 | 10.77 | 5.85 |
| Is routine heart rhythm assessment recommended for individuals aged 65 and older? | 69, 61, 59, 15, 42 | 0.5079 | 0.4918 | 0.5079 | 0.8571 | 0.9346 | 0.8942 | 0.8196 | 10 | 10 | 10 | 0.2327 | 11.33 | 6.09 |
| Which heart failure medication is recommended for AF patients regardless of LVEF? | 10, 26, 67, 66, 68 | 0.2778 | 0.1765 | 0.2778 | 0.7476 | 0.8654 | 0.8022 | 0.6102 | 9 | 10 | 10 | 0.242 | 10.46 | 5.07 |
| What should be done if a left atrial thrombus is detected prior to a planned cardioversion? | 42, 50, 46, 68, 67 | 0.1118 | 0.0 | 0.0621 | 0.7013 | 0.8083 | 0.751 | 0.4416 | 9 | 10 | 10 | 0.1903 | 24.13 | 10.2 |
| Name two non-cardiac conditions associated with trigger-induced AF according to Table 14. | 4, 16, 17, 11, 45 | 0.4878 | 0.4103 | 0.3902 | 0.8679 | 0.8149 | 0.8406 | 0.7494 | 10 | 10 | 10 | 0.2151 | 10.47 | 2.77 |
| What is the recommended weight loss target for obese individuals with AF? | 12, 66, 26, 11, 27 | 0.6 | 0.4286 | 0.5333 | 0.8503 | 0.8811 | 0.8654 | 0.6459 | 10 | 1 | 1 | 0.1895 | 8.27 | 3.14 |
| Can a physician use a single-lead ECG to provide a definite diagnosis of AF? | 11, 15, 61, 69, 12 | 0.2239 | 0.0758 | 0.1343 | 0.7684 | 0.8374 | 0.8015 | 0.7556 | 9 | 10 | 10 | 0.205 | 18.17 | 8.97 |
| What are the symptoms of AF listed in Figure 1? | 16, 15, 8, 20, 45 | 0.1333 | 0.0 | 0.1333 | 0.7366 | 0.7023 | 0.719 | 0.3274 | 0 | 0 | 1 | 0.2238 | 8.32 | 2.52 |
| Is it recommended to use a bleeding risk score to decide whether to withhold OAC? | 35, 36, 20, 67, 12 | 0.3261 | 0.2667 | 0.3043 | 0.7834 | 0.9129 | 0.8432 | 0.7944 | 1 | 1 | 1 | 0.1967 | 15.11 | 8.14 |
| What is the definition of "Permanent AF" according to the guidelines? | 14, 13, 8, 10, 9 | 0.875 | 0.8261 | 0.875 | 0.9205 | 0.954 | 0.937 | 0.8179 | 10 | 10 | 10 | 0.185 | 8.34 | 4.44 |
| For patients undergoing cardiac surgery, when is surgical closure of the LAA recommended? | 12, 48, 81, 11, 13 | 0.4507 | 0.3768 | 0.2817 | 0.8043 | 0.9007 | 0.8498 | 0.6067 | 10 | 10 | 10 | 0.1886 | 12.54 | 7.26 |

## Section 7. Results and Analysis

This section presents the evaluation results for the **Phi-3-Mini-4k-Instruct** RAG system, tested on **20 unseen clinical queries** drawn from the 2024 ESC Guidelines on Atrial Fibrillation.

---

### 7.1 Evaluation Metrics Summary

**Lexical Metrics**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| ROUGE-1 | 0.3860 | 0.1997 | 0.1118 | 0.8750 |
| ROUGE-2 | 0.2663 | 0.2154 | 0.0000 | 0.8261 |
| ROUGE-L | 0.3304 | 0.2101 | 0.0621 | 0.8750 |

**Semantic Metrics**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| BERTScore Precision | 0.7961 | 0.0629 | 0.6704 | 0.9205 |
| BERTScore Recall | 0.8568 | 0.0641 | 0.7023 | 0.9540 |
| BERTScore F1 | 0.8244 | 0.0578 | 0.7190 | 0.9370 |
| Semantic Similarity | 0.6282 | 0.1534 | 0.3114 | 0.8196 |

**RAG-Specific Metrics (LLM-as-Judge, /10)**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| Faithfulness | 8.75 | 2.86 | 0.0 | 10.0 |
| Answer Relevancy | 8.60 | 3.42 | 0.0 | 10.0 |
| Context Recall | 8.50 | 3.30 | 1.0 | 10.0 |

**Performance**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| Retrieval Time (s) | 0.2153 | 0.0484 | 0.1616 | 0.4035 |
| Generation Time (s) | 12.934 | 4.381 | 8.270 | 24.130 |
| Tokens/sec | 5.968 | 2.185 | 2.520 | 10.200 |

---

### 7.2 Analysis

Phi-3-Mini-4k-Instruct produced **surprisingly competitive results** on this updated full-metrics evaluation, which stands in contrast to the earlier preliminary assessment. Its Faithfulness score of **8.75/10** was the highest of all three models, indicating that when Phi-3 does generate a response, it tends to stay grounded in the retrieved context. Answer Relevancy (8.60/10) and Context Recall (8.50/10) were also within a comparable range to the other models.

On lexical metrics, Phi-3 scored a ROUGE-L mean of **0.3304** — the lowest among the three — reflecting a tendency toward more verbose or paraphrased responses that diverge in surface form from reference answers. Its BERTScore F1 of **0.8244** remains solid, however, confirming that semantic content is largely preserved even when the phrasing differs.

A key strength of Phi-3 is its **retrieval speed**: at just **0.2153 seconds** average retrieval time, it is the fastest at the document-fetching stage. Generation averages **12.93 seconds**, which is moderate, though with a high standard deviation (±4.38s) — indicating some queries triggered significantly longer outputs.

---

### 7.3 Limitations
- Phi-3-Mini's smaller parameter count (3.8B) makes it more sensitive to prompt structure; performance may degrade on more complex multi-part clinical queries.
- The high std in Generation Time (±4.38s) suggests inconsistent response length, which may affect user experience in a deployed interface.
- Evaluation was conducted on 20 queries; a larger test set would improve statistical confidence.

---

## Section 7 (Continued). Cross-Model Comparative Analysis

The following analysis compares all three RAG systems — **Llama-3-8B-Instruct**, **Phi-3-Mini-4k-Instruct**, and **Qwen3-4B-Instruct** — evaluated on the same 20 unseen clinical queries from the 2024 ESC Guidelines on Atrial Fibrillation.

---

### Cross-Model Metrics Overview

| Metric | Llama-3-8B | Phi-3-Mini | Qwen3-4B |
|--------|-----------|------------|----------|
| ROUGE-1 | **0.4564** | 0.3860 | 0.3921 |
| ROUGE-2 | **0.3366** | 0.2663 | 0.2877 |
| ROUGE-L | **0.4072** | 0.3304 | 0.3651 |
| BERTScore F1 | **0.8351** | 0.8244 | 0.8187 |
| Semantic Similarity | **0.6647** | 0.6282 | 0.6164 |
| Faithfulness (/10) | 8.05 | **8.75** | 8.50 |
| Answer Relevancy (/10) | **8.90** | 8.60 | **8.90** |
| Context Recall (/10) | **9.00** | 8.50 | 8.30 |
| Tokens/sec | 4.43 | 5.97 | **7.90** |

---

### Key Findings

**1. Llama-3 leads on lexical and semantic similarity.**
Llama-3-8B achieved the highest scores on every lexical metric (ROUGE-1: 0.4564, ROUGE-L: 0.4072) and the highest BERTScore F1 (0.8351) and Semantic Similarity (0.6647). This reflects that its responses most closely match reference answers both in surface form and semantic meaning — a result of its larger 8B parameter backbone and strong instruction-following via the Llama chat template.

**2. Phi-3 leads on Faithfulness.**
Phi-3-Mini achieved the highest Faithfulness score (8.75/10), meaning its responses — when generated — were more consistently grounded in the retrieved context than either competing model. This is notable given its smaller size and suggests that its strict clinical system prompt is effective at preventing the model from introducing unsupported claims.

**3. Qwen3 leads on generation speed.**
At 7.90 tokens/sec, Qwen3 generates responses nearly twice as fast as Llama-3 (4.43 tokens/sec), making it the most practical choice for latency-sensitive deployment. Despite this speed advantage, it maintains fully competitive Answer Relevancy (8.90/10) and solid BERTScore F1 (0.8187).

**4. All three models show high RAG-judge scores, but with variance.**
The large standard deviations in Faithfulness, Answer Relevancy, and Context Recall (ranging from ±2.76 to ±3.64 across models) indicate that all three systems encounter edge-case queries where performance degrades — typically on highly specific procedural or table-dense questions within the ESC Guidelines. This points to a shared retrieval limitation rather than a model-specific generation weakness.

---

### Overall Recommendation

All three models are viable for a clinical RAG system grounded in the 2024 ESC Guidelines, each with a distinct strength profile:

- **Llama-3-8B-Instruct** — Best for accuracy-critical applications where lexical and semantic fidelity to reference answers is the priority.
- **Phi-3-Mini-4k-Instruct** — Best for hallucination-sensitive applications where grounding in retrieved context is non-negotiable, with the added benefit of fast retrieval.
- **Qwen3-4B-Instruct** — Best for real-time deployment where generation speed is essential without significantly sacrificing answer quality.

---

### Future Work
- Extend the evaluation set beyond 20 queries to improve statistical reliability, particularly for high-variance metrics like Context Recall.
- Explore domain-specific fine-tuning on ESC guideline QA pairs to raise the performance floor on edge-case queries across all three models.
- Investigate ensemble or routing strategies that direct queries to the most appropriate model based on query complexity or retrieval confidence.
- Benchmark all three models under identical GPU-optimized inference conditions (e.g., vLLM) for a controlled latency comparison.